# 02 时域标量特征

三个核心特征：
- **振幅包络（Amplitude Envelope, AE）**：每帧内绝对值的最大值，刻画「峰值轮廓」
- **RMS（Root Mean Square）**：每帧内平方平均再开方，刻画 RMS 幅度水平
- **过零率（Zero Crossing Rate, ZCR）**：每帧内发生符号变化的相邻样本对所占比例，刻画符号变化密度

在四类音频上并排对比：钢琴、打击乐、弦乐、人声。

## 1. 环境自检与配置

In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from IPython import display as ipydisplay
from pathlib import Path
import warnings

import librosa
import librosa.display

# 统一参数
SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)

print(f"librosa={librosa.__version__}")

## 2. 加载四类对比音频

In [ ]:
def load_audio_slice(path, start_sec=0.0, duration_sec=3.0, sample_rate=SAMPLE_RATE):
    """加载音频切片，统一单声道 22050 Hz；返回相对于切片起点的时间轴（0 起始）"""
    samples, sr = librosa.load(path, sr=sample_rate, mono=True, offset=start_sec, duration=duration_sec)
    time_axis = np.linspace(0, len(samples) / sr, len(samples), endpoint=False)
    return time_axis, samples, sr

audio_sources = {
    "钢琴": (DATASET_DIR / "piano_solo.wav", 10.0),
    "打击乐": (DATASET_DIR / "orch_perc.wav", 2.0),
    "小提琴": (DATASET_DIR / "zhao_violin_wet.wav", 15.0),
    "人声": (DATASET_DIR / "xiaohetang_vox.wav", 30.0),
}

loaded_audio = {}
for label, (path, start) in audio_sources.items():
    if not path.exists():
        warnings.warn(f"跳过缺失文件：{path}")
        continue
    t, samples, sr = load_audio_slice(path, start_sec=start, duration_sec=3.0)
    loaded_audio[label] = (t, samples)
    print(f"{label:12s}: {len(samples):6d} samples | max={np.max(np.abs(samples)):.4f}")

print(f"\n已加载 {len(loaded_audio)} 段音频，统一 {SAMPLE_RATE} Hz 单声道")

## 3. 时域标量特征的计算

三个特征共享同一套分帧参数：
- **帧长（frame_length）**：每帧多少采样点
- **帧移（hop_length）**：相邻帧错开多少采样点

为严格对齐边界帧，下面先在波形两侧做相同的零填充，再以 `center=False` 对同一批帧计算 AE、RMS 与 ZCR。这样避免了 AE/RMS 的零填充与 `zero_crossing_rate` 默认 edge padding 混用。


In [ ]:
def make_zero_padded_frames(samples, frame_length=2048, hop_length=512):
    """在两侧零填充后生成共同帧矩阵，形状为 (frame_length, n_frames)。"""
    pad_length = frame_length // 2
    padded = np.pad(samples, (pad_length, pad_length), mode="constant")
    return librosa.util.frame(padded, frame_length=frame_length, hop_length=hop_length)

def compute_amplitude_envelope(samples, frame_length=2048, hop_length=512):
    frames = make_zero_padded_frames(samples, frame_length, hop_length)
    return np.max(np.abs(frames), axis=0)

def compute_rms_energy(samples, frame_length=2048, hop_length=512):
    frames = make_zero_padded_frames(samples, frame_length, hop_length)
    return np.sqrt(np.mean(frames ** 2, axis=0))

def compute_zero_crossing_rate(samples, frame_length=2048, hop_length=512):
    frames = make_zero_padded_frames(samples, frame_length, hop_length)
    crossings = np.signbit(frames[:-1]) != np.signbit(frames[1:])
    return np.mean(crossings, axis=0)

def make_time_axis_for_frames(n_frames, sr=SAMPLE_RATE, hop_length=512):
    return librosa.frames_to_time(np.arange(n_frames), sr=sr, hop_length=hop_length)


### 3.1 默认参数下的特征对比

使用 `frame_length=2048`，`hop_length=512`，在四类音频上同时画出：
- 波形（灰色半透明）
- AE（蓝色，峰值轮廓）
- RMS（绿色，RMS 幅度）
- ZCR（红色，过零率）

注意：AE 和 RMS 的量纲与振幅一致，ZCR 是 [0, 1] 范围内的无量纲符号变化比例。

In [ ]:
FRAME_LENGTH = 2048
HOP_LENGTH = 512

features = {}
for label, (t, samples) in loaded_audio.items():
    ae = compute_amplitude_envelope(samples, FRAME_LENGTH, HOP_LENGTH)
    rms = compute_rms_energy(samples, FRAME_LENGTH, HOP_LENGTH)
    zcr = compute_zero_crossing_rate(samples, FRAME_LENGTH, HOP_LENGTH)
    assert len(ae) == len(rms) == len(zcr)
    time_frames = make_time_axis_for_frames(len(ae), sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
    features[label] = {
        "time_wave": t, "samples": samples, "time_frames": time_frames,
        "ae": ae, "rms": rms, "zcr": zcr,
    }

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey="row")
row_labels = ["振幅包络", "RMS 振幅", "过零率"]
row_colors = ["C0", "C2", "C3"]
row_keys = ["ae", "rms", "zcr"]

for col, (label, feat) in enumerate(features.items()):
    for row, (rlabel, rcolor, rkey) in enumerate(zip(row_labels, row_colors, row_keys)):
        ax = axes[row, col]
        if rkey != "zcr":
            ax.plot(feat["time_wave"], np.abs(feat["samples"]), color="0.75", alpha=0.55, lw=0.5)
        ax.plot(feat["time_frames"], feat[rkey], color=rcolor, lw=1.5, label=rlabel)
        if row == 0:
            ax.set_title(label, fontsize=12, fontweight="bold")
        if col == 0:
            ax.set_ylabel(rlabel, fontsize=10)
        ax.set_xlabel("时间 (s)", fontsize=9)
        ax.set_xlim(feat["time_wave"][0], feat["time_wave"][-1])
        ax.axhline(0, color="black", ls="--", lw=0.4, alpha=0.4)

axes[0, 0].set_ylim(0, 0.4)
axes[1, 0].set_ylim(0, 0.4)
axes[2, 0].set_ylim(0, 0.5)

plt.suptitle(f"时域标量特征（frame_length={FRAME_LENGTH}, hop_length={HOP_LENGTH}）", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "time_domain_features_default.png", dpi=600, bbox_inches="tight")
plt.show()


### 3.2 观察与解读

这四段选定切片在 AE、RMS 与 ZCR 上呈现不同的局部形态。AE 只看峰值，对孤立尖峰敏感；RMS 综合整帧振幅；ZCR 是无量纲的符号变化比例，与振幅大小不在同一量纲。

这些观察只描述当前片段，不能把某个数值范围当作钢琴、打击乐、小提琴或人声类别的普遍边界。要形成分类结论，需要更多录音和独立测试数据。


## 4. 参数实验：帧长与帧移的权衡

同一特征，用不同时间尺度去看，会得到不同的曲线。
这里以 **人声** 为例，比较三种配置：
- **短帧** `frame_length=1024, hop_length=256`（约 46 ms / 12 ms）：时间精度高，适合捕捉瞬态
- **默认** `frame_length=2048, hop_length=512`（约 93 ms / 23 ms）：中等时间尺度
- **长帧** `frame_length=4096, hop_length=1024`（约 186 ms / 46 ms）：更平滑，适合长时趋势

In [ ]:
param_sets = [
    ("短帧 (1024/256)", 1024, 256),
    ("默认 (2048/512)", 2048, 512),
    ("长帧 (4096/1024)", 4096, 1024),
]

t_vox, samples_vox = loaded_audio["人声"]

fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True, sharey="row")
for col, (pname, flen, hlen) in enumerate(param_sets):
    ae = compute_amplitude_envelope(samples_vox, flen, hlen)
    rms = compute_rms_energy(samples_vox, flen, hlen)
    zcr = compute_zero_crossing_rate(samples_vox, flen, hlen)
    t_frames = make_time_axis_for_frames(len(ae), sr=SAMPLE_RATE, hop_length=hlen)

    for row, (feat, rlabel, rcolor) in enumerate(zip([ae, rms, zcr], row_labels, row_colors)):
        ax = axes[row, col]
        if row < 2:
            ax.plot(t_vox, np.abs(samples_vox), color="0.75", alpha=0.6, lw=0.5)
        ax.plot(t_frames, feat, color=rcolor, lw=1.3)
        if row == 0:
            ax.set_title(pname, fontsize=11)
        if col == 0:
            ax.set_ylabel(rlabel, fontsize=10)
        ax.set_xlabel("时间 (s)", fontsize=9)
        ax.set_xlim(t_vox[0], t_vox[-1])

axes[0, 0].set_ylim(0, 0.3)
axes[1, 0].set_ylim(0, 0.3)
axes[2, 0].set_ylim(0, 0.7)

plt.suptitle("人声片段：帧长与帧移的影响", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "time_domain_features_param_vocal.png", dpi=600, bbox_inches="tight")
plt.show()


**参数实验的启示**：短帧保留更快的局部变化，长帧提供更平滑的统计量。这里同时改变了帧长与帧移，因此曲线平滑程度和采样密度都会变化；若要单独研究某一个因素，应固定另一个参数。


## 5. 逐帧峰值因子

对幅度为 $A$ 的纯正弦波，`RMS = A / sqrt(2)`，峰值因子 `peak / RMS = sqrt(2)`。标准做法是先在每一帧计算 `AE_t / RMS_t`，再对有效帧汇总；`平均 AE / 平均 RMS` 一般不等于平均峰值因子。


In [ ]:
print(f"{'音频片段':15s} {'平均 AE':>10s} {'平均 RMS':>10s} {'逐帧峰值因子中位数':>20s}")
print("-" * 65)
for label, feat in features.items():
    valid = feat["rms"] > 1e-8
    crest_per_frame = feat["ae"][valid] / feat["rms"][valid]
    print(
        f"{label:15s} {np.mean(feat['ae']):10.4f} {np.mean(feat['rms']):10.4f} "
        f"{np.median(crest_per_frame):20.2f}"
    )

print(f"\n理论参考：纯正弦波 crest factor = sqrt(2) ≈ {np.sqrt(2):.3f}")


## 6. 单帧放大：「每种特征一帧一个标量」的计算过程

取人声中中心约在 0.5 s 的某一帧，把该帧的波形、AE、RMS、ZCR 计算过程可视化。
图中每种特征都由同一个短时窗口得到一个局部统计量，而不是对整段音频取平均。

In [ ]:
# 从与前面特征完全相同的两侧零填充帧矩阵中取一帧
target_time_sec = 0.5
target_frame_idx = int(np.round(target_time_sec * SAMPLE_RATE / HOP_LENGTH))
shared_frames = make_zero_padded_frames(samples_vox, FRAME_LENGTH, HOP_LENGTH)
frame_samples = shared_frames[:, target_frame_idx]
frame_center_sample = target_frame_idx * HOP_LENGTH
frame_start_sample = frame_center_sample - FRAME_LENGTH // 2
frame_end_sample = frame_start_sample + FRAME_LENGTH
frame_time_ms = (np.arange(FRAME_LENGTH) - FRAME_LENGTH // 2) / SAMPLE_RATE * 1000

ae_val = np.max(np.abs(frame_samples))
rms_val = np.sqrt(np.mean(frame_samples ** 2))
crossing_idx = np.flatnonzero(np.signbit(frame_samples[:-1]) != np.signbit(frame_samples[1:])) + 1
zcr_val = len(crossing_idx) / max(len(frame_samples) - 1, 1)

# 数值应与前面同一索引处的三条特征完全一致
assert np.isclose(ae_val, features["人声"]["ae"][target_frame_idx])
assert np.isclose(rms_val, features["人声"]["rms"][target_frame_idx])
assert np.isclose(zcr_val, features["人声"]["zcr"][target_frame_idx])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_vox, samples_vox, color="gray", lw=0.8)
axes[0].axvspan(max(0, frame_start_sample) / SAMPLE_RATE,
                min(len(samples_vox), frame_end_sample) / SAMPLE_RATE,
                color="0.5", alpha=0.25, label="选定帧")
axes[0].set_title("完整人声波形（3 s）")
axes[0].set_xlabel("时间 (s)")
axes[0].set_ylabel("振幅")
axes[0].legend()

axes[1].plot(frame_time_ms, frame_samples, color="0.3", lw=1.0, label="帧内波形")
axes[1].axhline(ae_val, color="black", ls="-", lw=1.2, label=f"AE = {ae_val:.4f}")
axes[1].axhline(rms_val, color="0.4", ls="--", lw=1.2, label=f"RMS = {rms_val:.4f}")
axes[1].scatter(frame_time_ms[crossing_idx], np.zeros_like(crossing_idx), marker="x", s=18,
                color="0.15", label=f"过零位置（ZCR={zcr_val:.4f}）")
axes[1].set_title(
    f"第 {target_frame_idx} 帧：中心 {frame_center_sample/SAMPLE_RATE:.3f} s，"
    f"窗长 {FRAME_LENGTH/SAMPLE_RATE*1000:.0f} ms"
)
axes[1].set_xlabel("相对帧中心的时间 (ms)")
axes[1].set_ylabel("振幅")
axes[1].legend(loc="lower right", fontsize=8)
axes[1].axhline(0, color="black", lw=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "single_frame_zoom.png", dpi=600, bbox_inches="tight")
plt.show()


## 7. 小结

1. AE、RMS、ZCR 使用同一批零填充帧，边界和时间轴严格对齐。
2. AE 与 RMS 是振幅量；ZCR 无量纲，因此不把 ZCR 当作振幅水平线叠加。
3. 峰值因子按帧计算后汇总，避免用“平均 AE / 平均 RMS”冒充标准 crest factor。
4. 当前四段项目内录音只用于比较片段差异，不支持乐器类别的泛化结论。

下一步（`03_stft_spectrogram.ipynb`）将把“每种特征一帧一个标量”扩展为“一帧一个频率向量”。


In [ ]:
print("本 Notebook 生成的图像文件：")
files = list(OUTPUT_FIG_DIR.glob("time_domain_*.png")) + [OUTPUT_FIG_DIR / "single_frame_zoom.png"]
for f in sorted(path for path in files if path.exists()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:8.1f} KB")